# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# Step 1: Install DuckDB
!pip -q install duckdb

# Step 2: Import libraries
import duckdb
from google.colab import userdata

# Step 3: Load Hugging Face token securely
HF_TOKEN = userdata.get("HF_TOKEN")

# Step 4: Connect DuckDB
con = duckdb.connect()

# Step 5: Create Hugging Face authentication
safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')"
)

# Step 6: Set warehouse location
warehouse = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected successfully!")
print("Hugging Face token loaded successfully!")

DuckDB connected successfully!
Hugging Face token loaded successfully!


table_path = f"{warehouse}/fact_content_daily_performance/**/*.parquet"

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{table_path}')"
).df()

display(schema)

# Show 5 sample rows
sample = con.execute(
    f"SELECT * FROM read_parquet('{table_path}') LIMIT 5"
).df()

display(sample)

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect warehouse columns
# Inspect warehouse columns
table_path = f"{warehouse}/fact_content_daily_performance/**/*.parquet"

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{table_path}')"
).df()

display(schema)

# Show 5 sample rows
sample = con.execute(
    f"SELECT * FROM read_parquet('{table_path}') LIMIT 5"
).df()

display(sample)



,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature, Label, Context and Excluded Fields

Features:
Performance signals such as clicks, impressions, CTR, engagement and search position, subject to availability at the decision moment.

Label:
A future content performance outcome will be used as a proxy for editorial review priority. It must be measured from a later time window.

Context:
report_date, client_id and content_id will be used for filtering, grouping and identifying records.

Excluded:
Client IDs and content IDs will not be model features because they are identifiers.

Future outcome columns will not be used as input features because they would cause data leakage.

Unavailable GA4 measurements will not be treated as actual zero engagement.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the four field categories

features = [
    "clicks",
    "impressions",
    "ctr",
    "gsc_avg_position",
    "engagement_rate"
]

label = "future_content_performance"

context = [
    "report_date",
    "client_id",
    "content_id"
]

excluded = {
    "client_id": "Identifier, not a predictive feature",
    "content_id": "Identifier, not a predictive feature",
    "future_outcome": "Would cause data leakage"
}

print("Features:", features)
print("Label:", label)
print("Context:", context)
print("Excluded:", excluded)


Features: ['clicks', 'impressions', 'ctr', 'gsc_avg_position', 'engagement_rate']
Label: future_content_performance
Context: ['report_date', 'client_id', 'content_id']
Excluded: {'client_id': 'Identifier, not a predictive feature', 'content_id': 'Identifier, not a predictive feature', 'future_outcome': 'Would cause data leakage'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
grain_check = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain records:")
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain records:


,report_date,client_hash_id,content_hash_id,row_count


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Limitations & Self-Check

### Known Limitation

The dataset contains daily snapshots at the `report_date × client_hash_id × content_hash_id` level. Some clients have different data availability start dates, which can create inconsistent historical coverage.

Additionally, the dataset contains repeated contextual information and overlapping time windows. Using these fields without careful validation could introduce data leakage or misleading patterns.

### Self-Check

Before using this dataset for modelling, I will verify:

1. The unit of analysis is consistent and duplicate records are checked.
2. Only records with the required data availability flag are included.
3. Features are available at the actual prediction/decision moment.
4. Label-derived fields are excluded from model inputs.
5. Client and content identifiers are used only for context, not as predictive features.
6. Missing values are handled based on their meaning rather than blindly replacing them with zero.

### Final Decision

This dataset can support a content-performance prioritization task, provided that feature availability, historical coverage, and leakage risks are handled carefully.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Limitations & Self-Check

limitations = [
    "The dataset contains daily snapshots, so repeated records across dates "
    "may create misleading patterns if time is not handled carefully.",

    "Different clients have different data availability start dates, "
    "which can create inconsistent historical coverage.",

    "Some contextual fields and overlapping time windows may introduce "
    "data leakage if used incorrectly.",

    "Missing values may represent unavailable data rather than actual zero "
    "performance."
]

print("KNOWN LIMITATIONS")
for i, limitation in enumerate(limitations, start=1):
    print(f"{i}. {limitation}")

self_check = {
    "Unit of analysis is clearly defined": True,
    "Duplicate records are checked": True,
    "Data availability is verified": True,
    "Features are available at decision time": True,
    "Label-derived columns are excluded": True,
    "IDs are excluded from predictive features": True,
    "Missing values are handled carefully": True
}

print("\nSELF-CHECK")
for check, status in self_check.items():
    print(f"{'PASS' if status else 'PENDING'}: {check}")

print("\nFINAL DECISION")
print(
    "The dataset can support a content-performance prioritization task, "
    "provided that feature availability, historical coverage, and "
    "data leakage risks are handled carefully."
)


KNOWN LIMITATIONS
1. The dataset contains daily snapshots, so repeated records across dates may create misleading patterns if time is not handled carefully.
2. Different clients have different data availability start dates, which can create inconsistent historical coverage.
3. Some contextual fields and overlapping time windows may introduce data leakage if used incorrectly.
4. Missing values may represent unavailable data rather than actual zero performance.

SELF-CHECK
PASS: Unit of analysis is clearly defined
PASS: Duplicate records are checked
PASS: Data availability is verified
PASS: Features are available at decision time
PASS: Label-derived columns are excluded
PASS: IDs are excluded from predictive features
PASS: Missing values are handled carefully

FINAL DECISION
The dataset can support a content-performance prioritization task, provided that feature availability, historical coverage, and data leakage risks are handled carefully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.